In [7]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-07-12 14:22


In [8]:
import requests
import pandas as pd
from geopy.distance import geodesic
from datetime import datetime 
from google.transit import gtfs_realtime_pb2

In [141]:
import os
import requests

API_KEY = os.environ["GTFSSweden3"]

URL = (
    "https://opendata.samtrafiken.se/gtfs-rt/sweden/VehiclePositions.pb"
    f"?key={API_KEY}"
)
print(API_KEY ) 

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

response = requests.get(URL, headers=headers)

print(response.status_code)
feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

print(len(feed.entity)) 
rows = []

for entity in feed.entity:

    if not entity.HasField("vehicle"):
        continue

    vehicle = entity.vehicle

    rows.append({

        "vehicle_id": vehicle.vehicle.id,

        "trip_id":
            vehicle.trip.trip_id
            if vehicle.HasField("trip") else None,

        "latitude":
            vehicle.position.latitude,

        "longitude":
            vehicle.position.longitude,

        "bearing":
            vehicle.position.bearing
            if vehicle.position.HasField("bearing") else None,

        "speed":
            vehicle.position.speed
            if vehicle.position.HasField("speed") else None,

        "timestamp":
            vehicle.timestamp

    })

samtrafiken = pd.DataFrame(rows)

f2277323909847378fb5951b57fbce03
403


DecodeError: Error parsing message

In [132]:
import requests

operator = "SL" 
API_KEY = os.environ["GTFSSweden3"]

url = "https://opendata.samtrafiken.se/gtfs-rt-sweden/sl/VehiclePositions.pb"

r = requests.get(url, params={"key": API_KEY})

print(r.status_code)
print(r.headers.get("Content-Type"))
print(r.content[:50])

200
application/octet-stream
b'\n\r\n\x032.0\x10\x00\x18\xc0\xf8\xce\xd2\x06\x12\\\n\x1148061783872513575"G\n\x15\n\x1114010000'


In [133]:
import requests
from google.transit import gtfs_realtime_pb2

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(r.content)

print("Antal entities:", len(feed.entity))



Antal entities: 846


In [134]:
rows = []

for entity in feed.entity:
    if not entity.HasField("vehicle"):
        continue

    v = entity.vehicle

    rows.append({
        "vehicle_id": v.vehicle.id,
        "trip_id": v.trip.trip_id if v.HasField("trip") else "",
        "route_id": v.trip.route_id if v.HasField("trip") else "",
        "lat": v.position.latitude,
        "lon": v.position.longitude,
    })

import pandas as pd

df = pd.DataFrame(rows)

print(df.route_id.nunique())
print(df.route_id.sort_values().unique()) 
df

12
['' '9011001001900000' '9011001023800000' '9011001054100000'
 '9011001054200000' '9011001055200000' '9011001055300000'
 '9011001075800000' '9011001078700000' '9011001098200000'
 '9011008000800000' '9011008001100000']


,vehicle_id,trip_id,route_id,lat,lon
0,9031001001004806,14010000724327343,,59.317875,18.021404
1,9031001001004825,14010000686179377,,59.353146,18.096296
2,9031001004505580,14010000721151217,,59.689217,18.441294
3,9031001003005037,14010000597114867,,59.119366,18.073509
4,9031001001001540,14010000717647527,,59.357189,18.102173
...,...,...,...,...,...
841,9031001004505568,14010000721163312,,59.856747,19.022490
842,9031001003005402,14010000722228364,,59.310249,18.198772
843,9031001003007210,,9011001078700000,59.189846,17.630060
844,9031001003004975,14010000722595319,,59.242023,18.104177


In [143]:
import folium

# Centrera kartan över Stockholm
m = folium.Map(
    location=[59.33, 18.10],
    zoom_start=10,
    tiles="OpenStreetMap"
)

for _, row in df.iterrows():

    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=3,
        popup=f"Vehicle: {row.vehicle_id}",
        tooltip=row.vehicle_id,
        color="blue",
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

m

In [49]:
boats = df[
    (df.lon > 18.2) &
    (df.lat > 59.15)
]

In [50]:
boats

,vehicle_id,trip_id,route_id,lat,lon
2,9031001004505580,14010000721147860,,59.759113,18.697060
8,9031001003005363,14010000691057366,,59.287361,18.472191
10,9031001001007183,14010000717645843,,59.370869,18.213039
25,9031001003005324,14010000722224087,,59.351231,18.278294
43,9031001004505572,14010000721160101,,60.053310,18.771187
...,...,...,...,...,...
840,9031001003007240,14010000691147233,,59.309509,18.469055
864,9031001004505110,14010000721147142,,59.962990,18.606718
870,9031001000500540,14010000728357151,,59.443317,18.886436
885,9031001004505109,14010000721160197,,60.029720,18.605907


In [85]:
rows = []

for entity in feed.entity:
    if not entity.HasField("vehicle"):
        continue

    v = entity.vehicle

    rows.append({
        "vehicle_id": v.vehicle.id,
        "trip_id": v.trip.trip_id if v.HasField("trip") else "",
        "route_id": v.trip.route_id if v.HasField("trip") else "",
        "lat": v.position.latitude,
        "lon": v.position.longitude,
    })

import pandas as pd

df = pd.DataFrame(rows)

print(df.route_id.nunique())
print(df.route_id.sort_values().unique()) 
df
boats = df[
    (df.lat >= 58.6) &
    (df.lat <= 59.9) &
    (df.lon >= 17.8) &
    (df.lon <= 19.4)
]
import folium

# Centrera kartan över Stockholm
m = folium.Map(
    location=[59.33, 18.10],
    zoom_start=10,
    tiles="OpenStreetMap"
)

for _, row in boats.iterrows():

    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=3,
        popup=f"Vehicle: {row.vehicle_id}",
        tooltip=row.vehicle_id,
        color="blue",
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

m

9
['' '9011001001900000' '9011001023800000' '9011001054200000'
 '9011001054400000' '9011001055200000' '9011001075800000'
 '9011001078700000' '9011001098200000']


In [83]:
## Nära Ornö   
near_orno = df[
    (df.lat.between(59.13, 59.18)) &
    (df.lon.between(18.38, 18.48))
]

near_orno

,vehicle_id,trip_id,route_id,lat,lon
37,9031001003005100,14010000723744632,,59.143723,18.434097


In [76]:
with ZipFile("sweden_api.zip") as z:
    trips = pd.read_csv(z.open("trips.txt"))
    routes = pd.read_csv(z.open("routes.txt"))

/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_5458/863778859.py:2: DtypeWarning: Columns (0,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(z.open("trips.txt"))


In [70]:
trip = trips[trips.trip_id == "14010000723744632"]
trip

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,shape_id,samtrafiken_internal_trip_number


In [71]:
routes.columns

Index(['route_id', 'agency_id', 'route_short_name', 'route_long_name',
       'route_type', 'route_desc'],
      dtype='object')

In [72]:
ferry_routes = routes[routes.route_type == 4]["route_id"]

ferries = df[df.route_id.isin(ferry_routes)]

In [73]:
ferries

,vehicle_id,trip_id,route_id,lat,lon


In [86]:
trip = trips[trips.trip_id == "14010000723744632"]
trip

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,shape_id,samtrafiken_internal_trip_number


In [89]:
trips.dtypes 

route_id                             object
service_id                            int64
trip_id                               int64
trip_headsign                       float64
trip_short_name                      object
direction_id                          int64
shape_id                              int64
samtrafiken_internal_trip_number      int64
dtype: object

In [90]:
trip = trips[trips.trip_id == 14010000723744632]

In [91]:
trips["trip_id"] = trips["trip_id"].astype(str)
df["trip_id"] = df["trip_id"].astype(str)

trip = trips[trips.trip_id == df.iloc[37]["trip_id"]]

In [92]:
print("Realtime trip_id:", df.iloc[37]["trip_id"])
print("Finns i trips.txt:", (trips["trip_id"].astype(str) == str(df.iloc[37]["trip_id"])).any())

Realtime trip_id: 14010000723744632
Finns i trips.txt: True


In [94]:
trip = trips[trips.trip_id.astype(str) == "14010000723744632"]

print(trip)

               route_id  service_id            trip_id  trip_headsign  \
84007  9011001083900000         637  14010000723744632            NaN   

      trip_short_name  direction_id             shape_id  \
84007             NaN             0  1014010000711837017   

       samtrafiken_internal_trip_number  
84007                                22  


In [95]:
trip.T

,84007
route_id,9011001083900000
service_id,637
trip_id,14010000723744632
trip_headsign,NaN
trip_short_name,NaN
direction_id,0
shape_id,1014010000711837017
samtrafiken_internal_trip_number,22


In [96]:
route_id = trip.iloc[0]["route_id"]

print("Route ID:", route_id)

Route ID: 9011001083900000


In [97]:
route = routes[routes.route_id == route_id]

route

,route_id,agency_id,route_short_name,route_long_name,route_type,route_desc
478,9011001083900000,505000000000000001,839,NaN,700,NaN


In [98]:
route.T

,478
route_id,9011001083900000
agency_id,505000000000000001
route_short_name,839
route_long_name,NaN
route_type,700
route_desc,NaN


In [99]:
routes["route_type"].value_counts().sort_index()

route_type
100       66
101      459
103      123
105       20
106     2871
401        7
700     3363
714       12
900       30
1000     103
1501     521
Name: count, dtype: int64

In [100]:
ferries = routes[routes["route_type"] == 700]

ferries[[
    "route_short_name",
    "route_long_name",
    "agency_id",
    "route_id"
]].sort_values("route_short_name")

,route_short_name,route_long_name,agency_id,route_id
5655,032,Malmö - Berlin,505000000000000166,9011166003200000
1374,1,NaN,505000000000000010,9011010000100000
1383,1,NaN,505000000000000010,9011010002100000
1387,1,NaN,505000000000000010,9011010003100000
3727,1,NaN,505000000000000020,9011020000100000
...,...,...,...,...
3179,X6,NaN,505000000000000014,9011014520600000
3185,X76,NaN,505000000000000014,9011014527600000
3186,X77,NaN,505000000000000014,9011014527700000
3194,X90,NaN,505000000000000014,9011014529000000


In [101]:
df = (
    df
    .merge(
        trips[["trip_id", "route_id"]],
        on="trip_id",
        how="left"
    )
    .merge(
        routes[[
            "route_id",
            "route_short_name",
            "route_type",
            "agency_id"
        ]],
        on="route_id",
        how="left"
    )
)

KeyError: 'route_id'

In [103]:
print(df.columns.tolist())
print(routes.columns.tolist())

['vehicle_id', 'trip_id', 'route_id', 'lat', 'lon']
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_desc']


In [104]:
print(routes.head())

              route_id           agency_id route_short_name route_long_name  \
0     9011001000000000  505000000000000001               40              SL   
1  9011001000000000-41  505000000000000001               41              SL   
2  9011001000000000-43  505000000000000001               43              SL   
3  9011001000000000-48  505000000000000001               48              SL   
4     9011001000100000  505000000000000001                1             NaN   

   route_type  route_desc  
0         106         NaN  
1         106         NaN  
2         106         NaN  
3         106         NaN  
4         700         NaN  


In [105]:
print(routes.dtypes)

route_id             object
agency_id             int64
route_short_name     object
route_long_name      object
route_type            int64
route_desc          float64
dtype: object


In [106]:
boats = df.merge(
    routes[["route_id", "route_short_name", "route_type", "agency_id"]],
    on="route_id",
    how="left"
)

boats[boats.route_type == 700]

,vehicle_id,trip_id,route_id,lat,lon,route_short_name,route_type,agency_id
51,9031001001004849,,9011001023800000,59.351517,18.151522,238,700.0,5.050000e+17
91,9031001003003584,,9011001098200000,59.470219,17.918287,41B,700.0,5.050000e+17
245,9031001003003540,,9011001055200000,59.423225,17.832230,552H,700.0,5.050000e+17
369,9031001003003056,,9011001054400000,59.423775,17.832603,544,700.0,5.050000e+17
482,9031001003003064,,9011001054200000,59.451603,17.829090,542,700.0,5.050000e+17
525,9031001003003480,,9011001075800000,59.184731,17.659843,758,700.0,5.050000e+17
589,9031001003006106,,9011001078700000,59.189987,17.629972,787,700.0,5.050000e+17


In [108]:
from zipfile import ZipFile
import pandas as pd

with ZipFile("sweden_api.zip") as z:
    agencies = pd.read_csv(z.open("agency.txt"))

agencies.head()

,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_fare_url
0,50,Samtrafiken i Sverige AB,https://www.resrobot.se/,Europe/Stockholm,sv,NaN
1,500000000000000043,Trafikverket RDB,https://www.resrobot.se/,Europe/Stockholm,sv,NaN
2,500000000000000107,Vy flygbussarna,https://www.resrobot.se/,Europe/Stockholm,sv,NaN
3,500000000000000109,109,https://www.resrobot.se/,Europe/Stockholm,sv,NaN
4,500000000000000110,110,https://www.resrobot.se/,Europe/Stockholm,sv,NaN


In [124]:
with ZipFile("sweden_api.zip") as z:
    agencies = pd.read_csv(
        z.open("agency.txt"),
        dtype=str
    )

    routes = pd.read_csv(
        z.open("routes.txt"),
        dtype=str
    )

    trips = pd.read_csv(
        z.open("trips.txt"),
        dtype=str
    )
boats = (
    df[
        ["vehicle_id", "trip_id", "route_id", "lat", "lon"]
    ]
    .drop_duplicates()
    .merge(
        routes[
            ["route_id", "agency_id", "route_short_name", "route_type"]
        ],
        on="route_id",
        how="left"
    )
    .merge(
        agencies[
            ["agency_id", "agency_name"]
        ],
        on="agency_id",
        how="left"
    )
)


In [125]:
print(boats["agency_id"].unique()[:10])
print(agencies["agency_id"].unique()[:10])

[nan '505000000000000001']
['50' '500000000000000043' '500000000000000107' '500000000000000109'
 '500000000000000110' '500000000000000114' '500000000000000115'
 '500000000000000117' '500000000000000123' '500000000000000130']


In [127]:
boats.to_csv("boats.csv") 


In [9]:
end_time = time.time()
duration = end_time - start_time
print(f"Started: {start_str}") 
print(f"Finished in {duration:.2f} seconds.")


Started: 2026-07-12 14:22
Finished in 0.99 seconds.
